In [124]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torch.nn as nn
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import GroupShuffleSplit, train_test_split
import pandas as pd
import pickle
import sys
import math

In [122]:
class CrossAttentionFusionModel(nn.Module):
    def __init__(self, gene_dim=978, chem_dim=1032, hidden_dim=64, num_heads=4, dropout=0.1):
        """
        Args:
            gene_dim (int): Number of features in the gene expression data (e.g., 978 for L1000).
            chem_dim (int): Number of features in the chemical descriptors (e.g., 1032 for Morgan + physchem).
            hidden_dim (int): Size of the shared hidden dimension for attention.
            num_heads (int): Number of attention heads.
            dropout (float): Dropout rate to prevent overfitting.
        """
        super().__init__()
        
        # projection layers to map gene and chem features in the same hidden space
        self.gene_encoder = nn.Sequential(
            nn.Linear(gene_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Drug Encoder
        self.drug_encoder = nn.Sequential(
            nn.Linear(chem_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Cross Attention
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        #self.gene_to_drug_attn = nn.MultiheadAttention(
        #    embed_dim=hidden_dim,
        #    num_heads=num_heads,
        #    dropout=dropout,
        #    batch_first=True
        #)

        # LayerNorms
        self.norm = nn.LayerNorm(hidden_dim)
        #self.norm_drug = nn.LayerNorm(hidden_dim)

        # Fusion Head
        #self.fusion = nn.Sequential(
        #    nn.Linear(hidden_dim * 2, 256),
        #    nn.ReLU(),
        #   nn.Dropout(dropout),
        #
        #    nn.Linear(256, 128),
        #    nn.ReLU(),
        #    nn.Dropout(dropout)
        #)

        # Regression Head
        #self.regressor = nn.Linear(128, 1)
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim*2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
    '''
    def forward(self, x_gene, x_drug):

        batch_size = x_gene.size(0)

        # Encode gene features
        # (B,978) -> (B,32*128)
        gene_tokens = self.gene_encoder(x_gene)

        # (B,4096) -> (B,32,128)
        gene_tokens = gene_tokens.view(
            batch_size,
            self.num_tokens,
            self.hidden_dim
        )

        # Encode drug features
        drug_tokens = self.drug_encoder(x_drug)

        drug_tokens = drug_tokens.view(
            batch_size,
            self.num_tokens,
            self.hidden_dim
        )

        # Drug attends to Gene
        drug_attended, drug_gene_weights = (
            self.drug_to_gene_attn(
                query=drug_tokens,
                key=gene_tokens,
                value=gene_tokens
            )
        )

        drug_tokens = self.norm_drug(
            drug_tokens + drug_attended
        )

        gene_attended, gene_drug_weights = (self.gene_to_drug_attn(query=gene_tokens,
                                                                   key=drug_tokens,
                                                                   value=drug_tokens))

        gene_tokens = self.norm_gene(
            gene_tokens + gene_attended
        )

        # Pool tokens
        gene_repr = gene_tokens.mean(dim=1)

        drug_repr = drug_tokens.mean(dim=1)

        # Fusion
        fused = torch.cat(
            [gene_repr, drug_repr],
            dim=1
        )

        fused = self.fusion(fused)
        pred = self.regressor(fused)

        return pred.squeeze(1)'''
    
    def forward(self, x_gene, x_drug):
        # 1. Embeddings erstellen -> Shape: (B, hidden_dim)
        gene_emb = self.gene_encoder(x_gene)
        drug_emb = self.drug_encoder(x_drug)

        # 2. Für nn.MultiheadAttention benötigen wir eine Sequenz-Dimension: (B, 1, hidden_dim)
        gene_token = gene_emb.unsqueeze(1)
        drug_token = drug_emb.unsqueeze(1)

        # 3. Eine einzige Cross-Attention reicht oft völlig aus! 
        # Hier schaut die Chemie (Query) auf die Biologie (Key/Value)
        attn_output, _ = self.cross_attn(
            query=drug_token,
            key=gene_token,
            value=gene_token
        )
        
        # Residual-Verbindung + Normierung
        drug_context = self.norm(drug_token + attn_output).squeeze(1)
        gene_context = gene_emb # wir behalten das biologische Original-Embedding bei

        # 4. Fusion durch Konkatination -> Shape: (B, hidden_dim * 2)
        fused = torch.cat([gene_context, drug_context], dim=1)

        # 5. Vorhersage
        pred = self.regressor(fused)
        return pred.squeeze(1)

In [125]:
class AdvancedSelfAndCrossAttentionModel(nn.Module):
    def __init__(self, gene_dim=978, chem_dim=1032, hidden_dim=64, num_heads=4, num_tokens=8, dropout=0.3):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        self.num_tokens = num_tokens
        
        # encoder: Projizieren die flachen Features in eine Token-Sequenz (Batch, num_tokens, hidden_dim)
        self.gene_encoder = nn.Sequential(
            nn.Linear(gene_dim, num_tokens * hidden_dim),
            nn.LayerNorm(num_tokens * hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.drug_encoder = nn.Sequential(
            nn.Linear(chem_dim, num_tokens * hidden_dim),
            nn.LayerNorm(num_tokens * hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Positional Encoding (Optional, aber hilfreich, damit die Attention die Token-IDs unterscheiden kann)
        self.gene_pos = nn.Parameter(torch.randn(1, num_tokens, hidden_dim))
        self.drug_pos = nn.Parameter(torch.randn(1, num_tokens, hidden_dim))

        # 2. SELF-ATTENTION: Jede Modalität kommuniziert erst mit sich selbst
        self.gene_self_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.drug_self_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        
        self.norm_self_gene = nn.LayerNorm(hidden_dim)
        self.norm_self_drug = nn.LayerNorm(hidden_dim)

        # 3. CROSS-ATTENTION: Chemie fragt Biologie ab
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.norm_cross = nn.LayerNorm(hidden_dim)

        # 4. REGRESSION HEAD
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x_gene, x_drug):
        batch_size = x_gene.size(0)

        # --- Schritt 1: In Tokens transformieren ---
        # Shape: (B, num_tokens, hidden_dim)
        gene_tokens = self.gene_encoder(x_gene).view(batch_size, self.num_tokens, self.hidden_dim)
        drug_tokens = self.drug_encoder(x_drug).view(batch_size, self.num_tokens, self.hidden_dim)
        
        # Positional Embedding addieren
        gene_tokens = gene_tokens + self.gene_pos
        drug_tokens = drug_tokens + self.drug_pos

        # --- Schritt 2: Echte Self-Attention (Intra-modal) ---
        # Biologie filtert eigenes Rauschen
        gene_self_out, _ = self.gene_self_attn(query=gene_tokens, key=gene_tokens, value=gene_tokens)
        gene_tokens = self.norm_self_gene(gene_tokens + gene_self_out)

        # Chemie ordnet funktionelle Gruppen aneinander an
        drug_self_out, _ = self.drug_self_attn(query=drug_tokens, key=drug_tokens, value=drug_tokens)
        drug_tokens = self.norm_self_drug(drug_tokens + drug_self_out)

        # --- Schritt 3: Cross-Attention (Inter-modal) ---
        # Chemie (Query) greift auf biologischen Kontext (Key/Value) zu
        cross_out, _ = self.cross_attn(query=drug_tokens, key=gene_tokens, value=gene_tokens)
        drug_context = self.norm_cross(drug_tokens + cross_out)

        # --- Schritt 4: Pooling & Fusion ---
        # Wir mitteln über die Token-Dimension (Mean Pooling)
        gene_repr = gene_tokens.mean(dim=1)  # Shape: (B, hidden_dim)
        drug_repr = drug_context.mean(dim=1) # Shape: (B, hidden_dim)

        # Konkatenerieren zu (B, hidden_dim * 2)
        fused = torch.cat([gene_repr, drug_repr], dim=1)

        # Vorhersage
        pred = self.regressor(fused)
        return pred.squeeze(1)

Der DataLoader kümmert sich darum, dass das Modell die Daten in Batches bekommt

In [2]:
!{sys.executable} -m pip install -q "pyarrow>=13.0.0"

df = pd.read_pickle(
    r"C:\Users\Juli\Documents\Master\Projekt Genomforschung\Datasets\harmonized_data.pkl"
)

In [108]:
# training data group split in train val
target = 'LN_IC50'
pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
X_genomic = df.filter(regex=r'.* \(.*\)').values.astype('float32')
X_chem = df[pharmacophores + list(df.columns[df.columns.str.startswith('Bit_')])].values.astype('float32')
y = df[target].values.astype('float32')

# scale data
gene_scaler = StandardScaler()
chem_scaler = StandardScaler()
X_genomic = gene_scaler.fit_transform(X_genomic)
X_chem = chem_scaler.fit_transform(X_chem)

# split data into train and test
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=11626)
idx_train_val, idx_test = next(gss_test.split(X_genomic, y, groups=df['DRUG_ID']))

# separate validation set from training set
groups_train_val = df['DRUG_ID'].iloc[idx_train_val]
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.11, random_state=11626)
idx_train, idx_val = next(gss_val.split(X_genomic[idx_train_val], y[idx_train_val], groups=groups_train_val))

idx_train = idx_train_val[idx_train]
idx_val = idx_train_val[idx_val]

In [91]:
# training data random split in train val
target = 'LN_IC50'
pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
X_genomic = df.filter(regex=r'.* \(.*\)').values.astype('float32')
X_chem = df[pharmacophores + list(df.columns[df.columns.str.startswith('Bit_')])].values.astype('float32')
y = df[target].values.astype('float32')
indices = np.arange(len(y))
# split data into train and test
# 10% Test, 90% Train+Val
#idx_train_val, idx_test = train_test_split(indices, test_size=0.1, random_state=11626, shuffle=True)
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=11626)
idx_train_val, idx_test = next(gss_test.split(X_genomic, y, groups=df['DRUG_ID']))

# 80/10/10 Split
idx_train, idx_val = train_test_split(idx_train_val, test_size=0.11, random_state=11626, shuffle=True)

X_genomic_train, X_genomic_val, X_genomic_test = X_genomic[idx_train], X_genomic[idx_val], X_genomic[idx_test]
X_chem_train, X_chem_val, X_chem_test = X_chem[idx_train], X_chem[idx_val], X_chem[idx_test]
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]

gene_scaler = StandardScaler()
chem_scaler = StandardScaler()

X_genomic_train = gene_scaler.fit_transform(X_genomic_train) #?
X_genomic_val = gene_scaler.transform(X_genomic_val)
X_genomic_test = gene_scaler.transform(X_genomic_test)

X_chem_train = chem_scaler.fit_transform(X_chem_train) #?
X_chem_val = chem_scaler.transform(X_chem_val)
X_chem_test = chem_scaler.transform(X_chem_test)

In [109]:
class DrugResponseDataset(Dataset):
    def __init__(self, X_gen, X_ch, labels, indices):
        self.X_genomic = torch.tensor(X_gen[indices], dtype=torch.float32)
        self.X_chem = torch.tensor(X_ch[indices], dtype=torch.float32)
        self.y = torch.tensor(labels[indices], dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X_genomic[idx], self.X_chem[idx], self.y[idx]

train_loader = DataLoader(DrugResponseDataset(X_genomic, X_chem, y, idx_train), batch_size=128, shuffle=False)
val_loader = DataLoader(DrugResponseDataset(X_genomic, X_chem, y, idx_val), batch_size=128, shuffle=False)
test_loader = DataLoader(DrugResponseDataset(X_genomic, X_chem, y, idx_test), batch_size=128, shuffle=False)

In [129]:
# initialize model, loss function and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AdvancedSelfAndCrossAttentionModel(gene_dim=X_genomic.shape[1], chem_dim=X_chem.shape[1], hidden_dim=128, num_heads=4, dropout=0.3).to(device)
model.to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=2e-5, weight_decay=1e-2) # weight decay for regularization, no overfitting
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3) # learning rate scheduler to reduce LR if validation loss plateaus

epochs = 30
best_val_rmse = float("inf")

print("Start Training")

for epoch in range(epochs):
    # training loop
    model.train()
    train_loss = 0.0
    for batch_genes, batch_chem, batch_y in train_loader:
        batch_genes, batch_chem, batch_y = batch_genes.to(device), batch_chem.to(device), batch_y.to(device)
        optimizer.zero_grad()
        predictions = model(batch_genes, batch_chem)#.unsqueeze(-1) # ensure predictions have shape (Batch, 1) for MSELoss
        predictions = predictions.view(-1) # ensure predictions have the right shape for MSELoss
        batch_y = batch_y.view(-1) # ensure target is the right shape for MSELoss
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch_genes.size(0)
    
    total_train_loss = train_loss / len(idx_train)
    
    # validation loop
    model.eval()
    val_preds, val_targets = [], []

    with torch.no_grad():
        for batch_genes, batch_chem, batch_y in val_loader:
            batch_genes, batch_chem, batch_y = batch_genes.to(device), batch_chem.to(device), batch_y.to(device)
            predictions = model(batch_genes, batch_chem)
            predictions = predictions.view(-1) # ensure predictions have the right shape for MSELoss

            val_preds.extend(predictions.cpu().numpy())
            val_targets.extend(batch_y.cpu().numpy())
    val_preds = np.array(val_preds)
    val_targets = np.array(val_targets)

    val_rmse = root_mean_squared_error(val_targets, val_preds)
    # to save the best model
    if val_rmse < best_val_rmse:
        round = epoch+1
        best_val_rmse = val_rmse
        torch.save(model.state_dict(), "best_model.pt")
    val_r2 = r2_score(val_targets, val_preds)

    scheduler.step(val_rmse) # update learning rate based on validation RMSE
    print(f"Epoch {epoch+1:02d}/{epochs} | Train RMSE: {np.sqrt(total_train_loss):.4f} | Val RMSE: {val_rmse:.4f} | Val R²: {val_r2:.4f}")

# final test
model.eval()
test_preds, test_targets = [], []

with torch.no_grad():
    for batch_genes, batch_chem, batch_y in test_loader:
        batch_genes, batch_chem, batch_y = batch_genes.to(device), batch_chem.to(device), batch_y.to(device)
        predictions = model(batch_genes, batch_chem)
        test_preds.extend(predictions.cpu().numpy())
        test_targets.extend(batch_y.cpu().numpy())

# convert lists to numpy arrays for metric calculations
test_preds, test_targets = np.array(test_preds), np.array(test_targets)

# Metrices
test_rmse = root_mean_squared_error(test_targets, test_preds)
test_r2 = r2_score(test_targets, test_preds)
#correlation = np.corrcoef(test_preds, test_targets)[0, 1] # pearson correlation as a simple measure of predictive performance in regression
print(f"Test RMSE: {test_rmse:.4f} | Test R²: {test_r2:.4f}")
print(f"Best model in epoch {round} with Val RMSE: {best_val_rmse:.4f}")

Start Training
Epoch 01/30 | Train RMSE: 2.3246 | Val RMSE: 2.9323 | Val R²: 0.0166
Epoch 02/30 | Train RMSE: 1.5026 | Val RMSE: 2.9862 | Val R²: -0.0198
Epoch 03/30 | Train RMSE: 1.4292 | Val RMSE: 2.9632 | Val R²: -0.0041
Epoch 04/30 | Train RMSE: 1.3743 | Val RMSE: 2.9587 | Val R²: -0.0011
Epoch 05/30 | Train RMSE: 1.3358 | Val RMSE: 2.9548 | Val R²: 0.0015
Epoch 06/30 | Train RMSE: 1.3165 | Val RMSE: 3.0050 | Val R²: -0.0327
Epoch 07/30 | Train RMSE: 1.3014 | Val RMSE: 3.0246 | Val R²: -0.0462
Epoch 08/30 | Train RMSE: 1.2925 | Val RMSE: 3.0511 | Val R²: -0.0646
Epoch 09/30 | Train RMSE: 1.2812 | Val RMSE: 3.0775 | Val R²: -0.0831
Epoch 10/30 | Train RMSE: 1.2755 | Val RMSE: 3.0983 | Val R²: -0.0978
Epoch 11/30 | Train RMSE: 1.2720 | Val RMSE: 3.1164 | Val R²: -0.1107
Epoch 12/30 | Train RMSE: 1.2670 | Val RMSE: 3.1292 | Val R²: -0.1199
Epoch 13/30 | Train RMSE: 1.2626 | Val RMSE: 3.1471 | Val R²: -0.1327
Epoch 14/30 | Train RMSE: 1.2638 | Val RMSE: 3.1545 | Val R²: -0.1380
Epoch 1

In [81]:
print(df['LN_IC50'].min(), df['LN_IC50'].max(), df['LN_IC50'].mean(), df['LN_IC50'].std())

-8.747724 13.091505 2.7423887961279756 2.8239460654063873
